In [26]:
import pandas as pd 
import numpy as np

from matplotlib import pyplot as plt
from sklearn.preprocessing import StandardScaler

from pandas import read_csv
import math

from keras.models import Sequential
from keras.layers import Dense
from keras.layers import LSTM, Flatten
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_squared_error
#from keras.callbacks import EarlyStopping
from keras.layers import ConvLSTM2D

import ipympl 
%matplotlib ipympl

file_path = '../Dataset/preprocessed_combined_data_weekday_3-2022_3-2023.csv'

df = pd.read_csv(file_path)

df.set_index('Date Time', inplace=True)

display(df)

,kW,Nantum,Season,Altimeter reading,DP Temp (F),DB Temp (F),Heat_day,Cool_day,DP_12h,DB_12h,DP_24h,DB_24h,Weekday_0,Weekday_1,Weekday_2,Weekday_3,Weekday_4,Weekday_5,Weekday_6
Date Time,,,,,,,,,,,,,,,,,,,
2022-03-24 00:00:00,121.500,0,0,30.11,16.0,58,0.072917,0.010417,14.0,72.0,24.0,61.0,0,0,0,1,0,0,0
2022-03-24 00:15:00,121.500,0,0,30.11,16.0,58,0.072917,0.010417,14.0,72.0,24.0,61.0,0,0,0,1,0,0,0
2022-03-24 00:30:00,119.248,0,0,30.11,16.0,58,0.072917,0.010417,14.0,72.0,24.0,61.0,0,0,0,1,0,0,0
2022-03-24 00:45:00,120.600,0,0,30.11,16.0,58,0.072917,0.010417,14.0,72.0,24.0,61.0,0,0,0,1,0,0,0
2022-03-24 01:00:00,123.300,0,0,30.11,16.0,56,0.093750,0.000000,12.0,74.0,23.0,58.0,0,0,0,1,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2023-03-22 22:45:00,117.448,1,0,30.03,34.0,49,0.166667,0.000000,26.0,57.0,38.0,46.0,0,0,1,0,0,0,0
2023-03-22 23:00:00,115.200,1,0,30.03,34.0,50,0.156250,0.000000,24.0,57.0,38.0,45.0,0,0,1,0,0,0,0
2023-03-22 23:15:00,118.348,1,0,30.03,34.0,50,0.156250,0.000000,24.0,57.0,38.0,45.0,0,0,1,0,0,0,0


In [27]:
data = pd.DataFrame(df['kW'])

display(data)

,kW
Date Time,
2022-03-24 00:00:00,121.500
2022-03-24 00:15:00,121.500
2022-03-24 00:30:00,119.248
2022-03-24 00:45:00,120.600
2022-03-24 01:00:00,123.300
...,...
2023-03-22 22:45:00,117.448
2023-03-22 23:00:00,115.200
2023-03-22 23:15:00,118.348


In [28]:
#Convert pandas dataframe to numpy array
dataset = data.values

display(dataset)

array([[121.5  ],
       [121.5  ],
       [119.248],
       ...,
       [118.348],
       [116.1  ],
       [115.652]])

In [29]:
#Standardize features by removing the mean and scaling to unit variance
# scaler=StandardScaler()
scaler = MinMaxScaler(feature_range=(0, 1))
dataset = scaler.fit_transform(dataset)

In [30]:
train_size = int(len(dataset) * 0.8)
test_size = len(dataset) - train_size
train, test = dataset[0:train_size,:], dataset[train_size:len(dataset),:]

In [31]:
def to_sequences(dataset, seq_size=1):
    x = []
    y = []

    for i in range(len(dataset)-seq_size):
        #print(i)
        window = dataset[i:(i+seq_size), 0]
        x.append(window) 
        y.append(dataset[i+seq_size, 0])
        
    return np.array(x),np.array(y)

In [32]:
seq_size = 96  # Number of time steps to look back 
#Larger sequences (look further back) may improve forecasting.
# 96 = 1 day. 15 min data and 24 * 4 = 96

trainX, trainY = to_sequences(train, seq_size)
testX, testY = to_sequences(test, seq_size)

trainX.shape, trainY.shape
# testX.shape, testY.shape

((27858, 96), (27858,))

In [33]:
print("Shape of training set: {}".format(trainX.shape))
print("Shape of test set: {}".format(testX.shape))

Shape of training set: (27858, 96)
Shape of test set: (6893, 96)


In [34]:
# Reshape input to be [samples, time steps, features]
trainX = np.reshape(trainX, (trainX.shape[0], 1, trainX.shape[1]))
testX = np.reshape(testX, (testX.shape[0], 1, testX.shape[1]))

In [35]:
print('Single LSTM with hidden Dense...')
model = Sequential()
model.add(LSTM(64, input_shape=(None, seq_size)))
# model.add(Dense(32))
model.add(Dense(1))
model.compile(loss='mean_squared_error', optimizer='adam')
#monitor = EarlyStopping(monitor='val_loss', min_delta=1e-3, patience=20, 
#                        verbose=1, mode='auto', restore_best_weights=True)
model.summary()
# print('Train...')

Single LSTM with hidden Dense...


Model: "sequential_2"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 lstm_2 (LSTM)               (None, 64)                41216     
                                                                 
 dense_2 (Dense)             (None, 1)                 65        
                                                                 
Total params: 41281 (161.25 KB)
Trainable params: 41281 (161.25 KB)
Non-trainable params: 0 (0.00 Byte)
_________________________________________________________________


In [36]:
model.fit(trainX, trainY, validation_data=(testX, testY),
          verbose=2, epochs=25)


# make predictions

trainPredict = model.predict(trainX)
testPredict = model.predict(testX)


Epoch 1/25
871/871 - 8s - loss: 0.0038 - val_loss: 7.3548e-04 - 8s/epoch - 9ms/step
Epoch 2/25
871/871 - 4s - loss: 0.0017 - val_loss: 3.8443e-04 - 4s/epoch - 4ms/step
Epoch 3/25
871/871 - 4s - loss: 0.0015 - val_loss: 5.4255e-04 - 4s/epoch - 4ms/step
Epoch 4/25
871/871 - 3s - loss: 0.0014 - val_loss: 4.4497e-04 - 3s/epoch - 4ms/step
Epoch 5/25
871/871 - 3s - loss: 0.0013 - val_loss: 3.8382e-04 - 3s/epoch - 4ms/step
Epoch 6/25
871/871 - 3s - loss: 0.0013 - val_loss: 5.4020e-04 - 3s/epoch - 4ms/step
Epoch 7/25
871/871 - 3s - loss: 0.0013 - val_loss: 5.0764e-04 - 3s/epoch - 4ms/step
Epoch 8/25
871/871 - 4s - loss: 0.0013 - val_loss: 3.2314e-04 - 4s/epoch - 4ms/step
Epoch 9/25
871/871 - 4s - loss: 0.0012 - val_loss: 3.7956e-04 - 4s/epoch - 4ms/step
Epoch 10/25
871/871 - 4s - loss: 0.0012 - val_loss: 4.3650e-04 - 4s/epoch - 5ms/step
Epoch 11/25
871/871 - 4s - loss: 0.0012 - val_loss: 3.2185e-04 - 4s/epoch - 4ms/step
Epoch 12/25
871/871 - 4s - loss: 0.0012 - val_loss: 2.9532e-04 - 4s/epoch 

In [37]:
# invert predictions back to prescaled values
#This is to compare with original input values
#SInce we used minmaxscaler we can now use scaler.inverse_transform
#to invert the transformation.
trainPredict = scaler.inverse_transform(trainPredict)
trainY = scaler.inverse_transform([trainY])
testPredict = scaler.inverse_transform(testPredict)
testY = scaler.inverse_transform([testY])

# calculate root mean squared error
trainScore = (mean_squared_error(trainY[0], trainPredict[:,0]))
print('Train Score: %.2f MSE' % (trainScore))

testScore = (mean_squared_error(testY[0], testPredict[:,0]))
print('Test Score: %.2f MSE' % (testScore))

Train Score: 404.62 MSE
Test Score: 122.14 MSE


In [38]:
from sklearn.metrics import r2_score

# Assuming y_train_pred and y_test_pred are the predicted values, and y_train and y_test are the true values
train_r2 = r2_score(trainY[0], trainPredict[:, 0])
test_r2 = r2_score(testY[0], testPredict[:, 0])

print("Train R^2:", train_r2)
print("Test R^2:", test_r2)

Train R^2: 0.976312455629052
Test R^2: 0.9751103785204116


In [39]:
# shift train predictions for plotting
#we must shift the predictions so that they align on the x-axis with the original dataset. 
trainPredictPlot = np.empty_like(dataset)
trainPredictPlot[:, :] = np.nan
trainPredictPlot[seq_size:len(trainPredict)+seq_size, :] = trainPredict

# shift test predictions for plotting
testPredictPlot = np.empty_like(dataset)
testPredictPlot[:, :] = np.nan
testPredictPlot[len(trainPredict)+(seq_size*2)+1:len(dataset)-1, :] = testPredict

# plot baseline and predictions
plt.plot(scaler.inverse_transform(dataset) ,label="Real kW", color="yellow")
plt.plot(trainPredictPlot, label="Predicted train set", color="red")
plt.plot(testPredictPlot, label="Predicted test set", color="blue")
plt.show()

ValueError: could not broadcast input array from shape (6893,1) into shape (6891,1)